[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/pml-f2026-notebooks/blob/main/class-demos/session08_linear_regression.ipynb)

# Session 8 deck code, assembled in slide order

**Session 8 · the in-class demo — runs on the free tier of Colab, nothing to install**

Generated by scripts/make_demo.py from the slides themselves, so it stays
honest about what the deck actually shows. Run it before class.

Each heading names the slide its cell accompanies; the outputs below were saved from a real run.

## Slide 4: Our Dataset: 60 House Sales

In [1]:
import numpy as np

rng = np.random.default_rng(42)   # fixed seed
n = 60
# size in 1000s of sqft, price in $1000s:
size  = rng.uniform(0.8, 3.5, n)
price = 60 + 120*size + rng.normal(0, 40, n)

print(np.round(size[:4], 3))
print(np.round(price[:4], 1))

[2.89  1.985 3.118 2.683]
[372.1 336.9 366.9 368.6]


## Slide 16: Normal Equation in Code — Real Run

In [2]:
beds  = rng.integers(1, 6, n).astype(float)
age   = rng.uniform(0, 50, n)
noise = rng.normal(0, 30, n)
y = 45 + 110*size + 18*beds - 1.2*age + noise

X_b = np.c_[np.ones(n), size, beds, age]
theta = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y
print(np.round(theta, 3))

[ 58.934 105.059  17.976  -1.439]


## Slide 17: Don’t Invert in Production — Use lstsq

In [3]:
sol, *_ = np.linalg.lstsq(X_b, y, rcond=None)
print(np.round(sol, 3))

[ 58.934 105.059  17.976  -1.439]


## Slide 26: From Scratch, Part 1: Loss and Gradient

In [4]:
def mse_loss(X, y, theta):
    """MSE of predictions X @ theta."""
    residuals = y - X @ theta
    return np.mean(residuals ** 2)

def gradient(X, y, theta):
    """Gradient of MSE wrt theta."""
    m = len(y)
    return (2 / m) * X.T @ (X @ theta - y)

## Slide 27: From Scratch, Part 2: The Training Loop

In [5]:
def gradient_descent(X, y, lr=0.1, epochs=1000):
    theta = np.zeros(X.shape[1])   # start at 0
    for ep in range(1, epochs + 1):
        theta -= lr * gradient(X, y, theta)
        if ep in {1, 5, 50, 200, 1000}:
            L = mse_loss(X, y, theta)
            print(f"ep {ep:>4}  loss {L:9.3f}  "
                  f"theta {np.round(theta, 3)}")
    return theta

## Slide 28: From Scratch, Part 3: Run It — Real Output

In [6]:
X1 = np.c_[np.ones(n), size]   # [1, size] again
theta_gd = gradient_descent(X1, price)

ep    1  loss  8675.232  theta [ 63.227 153.07 ]
ep    5  loss   896.108  theta [ 50.189 120.756]
ep   50  loss   894.965  theta [ 51.974 119.824]
ep  200  loss   894.744  theta [ 53.325 119.266]
ep 1000  loss   894.743  theta [ 53.417 119.228]


## Slide 29: Did It Converge? Check, Don’t Hope

In [7]:
g = gradient(X1, price, theta_gd)
print(f"||gradient|| = {np.linalg.norm(g):.2e}")

||gradient|| = 7.59e-09


## Slide 30: Moment of Truth: Us vs scikit-learn

In [8]:
from sklearn.linear_model import LinearRegression

X_sk = size.reshape(-1, 1)   # sklearn: 2-D X, no 1s
lin = LinearRegression().fit(X_sk, price)

print(np.round([lin.intercept_, *lin.coef_], 3))
print(np.round(theta_gd, 3))   # ours

[ 53.417 119.228]
[ 53.417 119.228]


## Slide 31: Tooling Spine: Reproducibility & Seeds

In [9]:
rng = np.random.default_rng(42)   # modern, local
np.random.seed(42)     # legacy global - avoid

# requirements.txt - pin your versions:
# numpy==2.3.4
# scikit-learn==1.7.2